# 📊 Notebook 03: Modeling Time Series

Notebook ini menjalankan proses pemodelan **time series forecasting** menggunakan PyCaret.

**Alur Kerja:**
1. Import library & definisi konstanta
2. Definisi kelas `EksperimenSaham` (OOP)
3. Eksekusi pipeline: Muat Data → Setup → Cari Model Terbaik → Visualisasi → Simpan

## Cell 1: Import Library

In [ ]:
import os
import json
import warnings
import pandas as pd
from pathlib import Path
from pycaret.time_series import (
    setup,
    compare_models,
    pull,
    save_model,
    plot_model,
)
from IPython.display import display

warnings.filterwarnings('ignore')

## Cell 2: Konstanta & Path

In [4]:
# Definisi path menggunakan pathlib
BASE_DIR = Path('..')
DATA_PROCESSED_DIR = BASE_DIR / 'Data' / 'Processed'
JSON_TICKER_PATH   = BASE_DIR / 'Data' / 'Raw' / 'tickers_us.json'

# Arahkan output langsung ke folder Monthly
MODEL_MONTHLY_DIR  = BASE_DIR / 'Models' / 'Monthly'
os.makedirs(MODEL_MONTHLY_DIR, exist_ok=True)

# Konstanta strategi
UKURAN_TRAINING = 180     # 15 tahun (180 bulan)
HORIZON_PREDIKSI = 12    # Prediksi 1 tahun ke depan
JUMLAH_FOLD = 5           

## Cell 3: Ambil Daftar Saham dari JSON

In [5]:
def ambil_daftar_saham(json_path: Path) -> list:
    """Membaca file JSON dan mengambil ticker yang statusnya available."""
    with open(json_path, 'r') as f:
        data = json.load(f)
    return [t['ticker'] for t in data['tickers'] if t['available']]

daftar_saham = ambil_daftar_saham(JSON_TICKER_PATH)

## Cell 4: Definisi Kelas `EksperimenSaham`

In [7]:
class EksperimenSaham:
    def __init__(self, ticker: str):
        self.ticker = ticker
        self.df = None
        self.model_terbaik = None

    def muat_data_bulanan(self, folder_path: Path):
        """Mengambil data khusus bulanan."""
        file_path = folder_path / f'{self.ticker}_monthly.parquet'
        self.df = pd.read_parquet(file_path)
        if 'Date' in self.df.columns:
            self.df.set_index('Date', inplace=True)
        if hasattr(self.df.index, 'tz'):
            self.df.index = self.df.index.tz_localize(None)
        self.df = self.df.asfreq('M')
        self.df.ffill(inplace=True)
        return self

    def jalankan_setup(self, horizon: int, ukuran_training: int):
        """Setup PyCaret dengan Log Transformation."""
        jumlah_potong = ukuran_training + horizon
        data_final = self.df.tail(jumlah_potong) if len(self.df) > jumlah_potong else self.df
        
        setup(
            data=data_final[['Close']],
            fh=horizon,
            fold=JUMLAH_FOLD,
            fold_strategy='sliding',
            seasonal_period=12,
            transform_target='log', # <--- Log Transform Aktif
            session_id=123,
            verbose=False,
        )
        return self

    def cari_model_terbaik(self):
        self.model_terbaik = compare_models(sort='MAE')
        return self

    def simpan_ke_monthly(self, output_dir: Path):
        """Simpan model ke folder Models/Monthly."""
        nama_file = output_dir / f'{self.ticker}_model_bulanan'
        save_model(self.model_terbaik, str(nama_file))
        return self

## Cell 5: Eksekusi Pipeline — AAPL

Menjalankan seluruh pipeline untuk saham **AAPL** (Apple Inc).

In [8]:
eksperimen_aapl = EksperimenSaham('AAPL')
(
    eksperimen_aapl
    .muat_data_bulanan(DATA_PROCESSED_DIR)
    .jalankan_setup(horizon=HORIZON_PREDIKSI, ukuran_training=UKURAN_TRAINING)
    .cari_model_terbaik()
    .simpan_ke_monthly(MODEL_MONTHLY_DIR)
)

print(f'\n>>> [SEL 5] Hasil Eksperimen AAPL (LOG):')
display(pull())
plot_model(eksperimen_aapl.model_terbaik, plot='forecast')

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
theta,Theta Forecaster,1.7806,1.4043,20.8845,23.5911,0.1388,0.1506,-3.2011,0.0140
auto_arima,Auto ARIMA,1.6911,1.3293,21.3945,23.8958,0.1383,0.1429,-3.5988,0.1240
arima,ARIMA,1.5817,1.3062,21.6718,25.7033,0.1406,0.1351,-3.6947,0.0260
exp_smooth,Exponential Smoothing,1.7257,1.3739,21.9837,24.9290,0.1446,0.1458,-4.3222,0.0180
ets,ETS,1.7257,1.3739,21.9841,24.9292,0.1446,0.1458,-4.3223,0.0220
omp_cds_dt,Orthogonal Matching Pursuit w/ Cond. Deseasonalize & Detrending,2.1497,1.6870,25.5556,29.1885,0.1709,0.1846,-5.5981,0.0380
naive,Naive Forecaster,2.1424,1.6330,25.6639,27.7790,0.1635,0.1862,-4.9200,0.6520
stlf,STLF,2.1282,1.6494,28.6159,31.8683,0.1878,0.1821,-6.2887,0.0140
rf_cds_dt,Random Forest w/ Cond. Deseasonalize & Detrending,2.3567,1.8754,28.6374,32.5901,0.1944,0.2021,-6.9262,0.0820
ada_cds_dt,AdaBoost w/ Cond. Deseasonalize & Detrending,2.3844,1.8827,29.0751,32.5864,0.1961,0.2052,-7.2337,0.0440


Transformation Pipeline and Model Successfully Saved

>>> [SEL 5] Hasil Eksperimen AAPL (LOG):


,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
theta,Theta Forecaster,1.7806,1.4043,20.8845,23.5911,0.1388,0.1506,-3.2011,0.014
auto_arima,Auto ARIMA,1.6911,1.3293,21.3945,23.8958,0.1383,0.1429,-3.5988,0.124
arima,ARIMA,1.5817,1.3062,21.6718,25.7033,0.1406,0.1351,-3.6947,0.026
exp_smooth,Exponential Smoothing,1.7257,1.3739,21.9837,24.929,0.1446,0.1458,-4.3222,0.018
ets,ETS,1.7257,1.3739,21.9841,24.9292,0.1446,0.1458,-4.3223,0.022
omp_cds_dt,Orthogonal Matching Pursuit w/ Cond. Deseasona...,2.1497,1.687,25.5556,29.1885,0.1709,0.1846,-5.5981,0.038
naive,Naive Forecaster,2.1424,1.633,25.6639,27.779,0.1635,0.1862,-4.92,0.652
stlf,STLF,2.1282,1.6494,28.6159,31.8683,0.1878,0.1821,-6.2887,0.014
rf_cds_dt,Random Forest w/ Cond. Deseasonalize & Detrending,2.3567,1.8754,28.6374,32.5901,0.1944,0.2021,-6.9262,0.082
ada_cds_dt,AdaBoost w/ Cond. Deseasonalize & Detrending,2.3844,1.8827,29.0751,32.5864,0.1961,0.2052,-7.2337,0.044


## Cell 6: Visualisasi & Simpan Model — AAPL

In [9]:
# Visualisasi hasil prediksi
eksperimen_aapl.tampilkan_visualisasi()

# Simpan model terbaik ke folder Models/
eksperimen_aapl.simpan_model(MODEL_OUTPUT_DIR)

AttributeError: 'EksperimenSaham' object has no attribute 'tampilkan_visualisasi'

In [9]:
## Cell 7: Loop Pemodelan Massal (Bulk Modeling)
# Menjalankan pipeline untuk semua ticker yang tersedia secara otomatis.

print(f">>> Memulai pemodelan massal untuk {len(daftar_saham)} saham...")

# Wadah untuk menyimpan ringkasan hasil
ringkasan_semua_model = []

for ticker in daftar_saham:
    try:
        # 1. Inisialisasi
        eksperimen = EksperimenSaham(ticker)
        
        # 2. Eksekusi Pipeline (Muat -> Setup -> Cari Model -> Simpan)
        (
            eksperimen
            .muat_data(DATA_PROCESSED_DIR)
            .jalankan_setup(horizon=HORIZON_PREDIKSI, ukuran_training=UKURAN_TRAINING)
            .cari_model_terbaik()
        )
        
        # 3. Simpan Model ke Folder Models/
        eksperimen.simpan_model(MODEL_OUTPUT_DIR)
        
        # 4. Catat hasil terbaik untuk ringkasan akhir
        best_metrics = pull().iloc[0] # Ambil baris pertama (juara)
        ringkasan_semua_model.append({
            'Ticker': ticker,
            'Model Terbaik': best_metrics['Model'],
            'MAE': best_metrics['MAE'],
            'RMSE': best_metrics['RMSE'],
            'R2': best_metrics['R2']
        })
        
        print(f">>> [SUKSES] {ticker} selesai diproses.\n")
        
    except Exception as e:
        print(f">>> [GAGAL] Terjadi error pada ticker {ticker}: {str(e)}\n")

# Tampilkan ringkasan akhir semua saham
df_ringkasan = pd.DataFrame(ringkasan_semua_model)
print("="*50)
print("RINGKASAN PEMODELAN SELURUH SAHAM")
print("="*50)
display(df_ringkasan)

>>> Memulai pemodelan massal untuk 30 saham...
>>> [AAPL] Data dimuat: 253 baris
>>> [AAPL] Memulai setup PyCaret...
    Data dipotong menjadi 192 baris terakhir
    Setup berhasil! (fh=12, fold=5)
>>> [AAPL] Sedang membandingkan algoritma...


,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
ets,ETS,1.7594,1.3986,21.2915,24.0332,0.1430,0.1492,-3.4215,0.0180
arima,ARIMA,1.6833,1.3493,21.7193,24.6736,0.1395,0.1451,-3.4848,0.0160
exp_smooth,Exponential Smoothing,1.8080,1.4373,22.0331,24.8419,0.1468,0.1539,-3.6933,0.0160
theta,Theta Forecaster,1.9292,1.4953,22.3852,24.7939,0.1469,0.1648,-3.7600,0.0080
omp_cds_dt,Orthogonal Matching Pursuit w/ Cond. Deseasonalize & Detrending,2.0529,1.5670,23.9444,25.9415,0.1554,0.1767,-4.3564,0.0380
auto_arima,Auto ARIMA,2.0586,1.5839,24.4335,26.9129,0.1614,0.1777,-4.2470,1.1960
gbr_cds_dt,Gradient Boosting w/ Cond. Deseasonalize & Detrending,2.2129,1.7003,24.4678,26.9607,0.1683,0.1922,-4.9537,0.0400
et_cds_dt,Extra Trees w/ Cond. Deseasonalize & Detrending,2.2180,1.6971,24.5375,26.8486,0.1683,0.1928,-4.9354,0.0740
ada_cds_dt,AdaBoost w/ Cond. Deseasonalize & Detrending,2.2447,1.7100,25.5255,27.6633,0.1714,0.1952,-5.0719,0.0420
naive,Naive Forecaster,2.1424,1.6330,25.6639,27.7790,0.1635,0.1862,-4.9200,0.0160


Transformation Pipeline and Model Successfully Saved
>>> [AAPL] Model tersimpan di: ../Models/AAPL_model_terbaik
>>> [SUKSES] AAPL selesai diproses.

>>> [MSFT] Data dimuat: 253 baris
>>> [MSFT] Memulai setup PyCaret...
    Data dipotong menjadi 192 baris terakhir
    Setup berhasil! (fh=12, fold=5)
>>> [MSFT] Sedang membandingkan algoritma...


,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
auto_arima,Auto ARIMA,0.9389,0.8201,24.5648,29.6979,0.0814,0.0819,-0.5229,3.9420
br_cds_dt,Bayesian Ridge w/ Cond. Deseasonalize & Detrending,0.9395,0.8603,26.1924,32.7116,0.0826,0.0863,-0.7208,0.0340
en_cds_dt,Elastic Net w/ Cond. Deseasonalize & Detrending,0.9452,0.8701,26.3551,33.0164,0.0835,0.0872,-0.7482,0.0780
llar_cds_dt,Lasso Least Angular Regressor w/ Cond. Deseasonalize & Detrending,0.9504,0.8740,26.4016,33.0935,0.0837,0.0875,-0.7629,0.0320
lasso_cds_dt,Lasso w/ Cond. Deseasonalize & Detrending,0.9506,0.8741,26.4039,33.0953,0.0837,0.0876,-0.7630,0.0300
ridge_cds_dt,Ridge w/ Cond. Deseasonalize & Detrending,0.9539,0.8639,26.5711,32.9001,0.0846,0.0878,-0.7146,0.0340
lr_cds_dt,Linear w/ Cond. Deseasonalize & Detrending,0.9540,0.8639,26.5728,32.9007,0.0846,0.0878,-0.7146,0.0320
omp_cds_dt,Orthogonal Matching Pursuit w/ Cond. Deseasonalize & Detrending,1.0403,0.8938,26.7580,31.9476,0.0881,0.0911,-0.6900,0.0640
huber_cds_dt,Huber w/ Cond. Deseasonalize & Detrending,1.0878,1.0084,30.4402,38.6463,0.0942,0.1012,-1.3270,0.0460
exp_smooth,Exponential Smoothing,1.1272,0.9658,30.4850,35.8794,0.1038,0.1013,-1.1322,0.0580


Transformation Pipeline and Model Successfully Saved
>>> [MSFT] Model tersimpan di: ../Models/MSFT_model_terbaik
>>> [SUKSES] MSFT selesai diproses.

>>> [GOOGL] Data dimuat: 253 baris
>>> [GOOGL] Memulai setup PyCaret...
    Data dipotong menjadi 192 baris terakhir
    Setup berhasil! (fh=12, fold=5)
>>> [GOOGL] Sedang membandingkan algoritma...


...

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
dt_cds_dt,Decision Tree w/ Cond. Deseasonalize & Detrending,1.4080,1.2546,32.5615,39.3280,0.1478,0.1448,-1.1339,0.0320
et_cds_dt,Extra Trees w/ Cond. Deseasonalize & Detrending,1.5022,1.3176,33.4899,40.2942,0.1555,0.1551,-1.4044,0.0740
ets,ETS,1.4630,1.2986,34.4804,41.4240,0.1543,0.1562,-1.2591,0.0180
exp_smooth,Exponential Smoothing,1.4639,1.2985,34.5092,41.4185,0.1545,0.1563,-1.2587,0.0160
gbr_cds_dt,Gradient Boosting w/ Cond. Deseasonalize & Detrending,1.5543,1.3557,35.4139,41.9475,0.1679,0.1616,-1.4682,0.0400
huber_cds_dt,Huber w/ Cond. Deseasonalize & Detrending,1.5749,1.3809,35.7789,42.3867,0.1673,0.1655,-1.7593,0.0340
omp_cds_dt,Orthogonal Matching Pursuit w/ Cond. Deseasonalize & Detrending,1.5820,1.3790,36.0215,42.8158,0.1617,0.1658,-1.7311,0.0300
ada_cds_dt,AdaBoost w/ Cond. Deseasonalize & Detrending,1.6157,1.3929,37.2695,43.6024,0.1741,0.1707,-1.7169,0.0420
theta,Theta Forecaster,1.5754,1.3707,37.5317,44.1739,0.1598,0.1684,-1.7020,0.0080
auto_arima,Auto ARIMA,1.5527,1.3843,37.6942,45.3717,0.1567,0.1677,-1.7400,1.0520


Transformation Pipeline and Model Successfully Saved
>>> [CRM] Model tersimpan di: ../Models/CRM_model_terbaik
>>> [SUKSES] CRM selesai diproses.

>>> [NFLX] Data dimuat: 253 baris
>>> [NFLX] Memulai setup PyCaret...
    Data dipotong menjadi 192 baris terakhir
    Setup berhasil! (fh=12, fold=5)
>>> [NFLX] Sedang membandingkan algoritma...


,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
huber_cds_dt,Huber w/ Cond. Deseasonalize & Detrending,1.4792,1.2592,11.1907,13.4294,0.2260,0.2140,-2.8757,0.0440
dt_cds_dt,Decision Tree w/ Cond. Deseasonalize & Detrending,1.5361,1.3144,11.3672,13.6616,0.2347,0.2298,-3.6072,0.0340
omp_cds_dt,Orthogonal Matching Pursuit w/ Cond. Deseasonalize & Detrending,1.5026,1.2833,11.4468,13.7340,0.2201,0.2112,-3.4948,0.0300
gbr_cds_dt,Gradient Boosting w/ Cond. Deseasonalize & Detrending,1.5504,1.3352,11.4747,13.9238,0.2530,0.2268,-3.4473,0.0420
ets,ETS,1.5206,1.3729,11.5200,14.5932,0.2431,0.2348,-3.0767,0.0160
rf_cds_dt,Random Forest w/ Cond. Deseasonalize & Detrending,1.5443,1.3366,11.8359,14.2890,0.2441,0.2241,-3.3497,0.0800
et_cds_dt,Extra Trees w/ Cond. Deseasonalize & Detrending,1.6714,1.4120,12.6728,15.0129,0.2622,0.2465,-4.2761,0.0780
auto_arima,Auto ARIMA,1.6317,1.4294,13.0291,15.8225,0.2697,0.2714,-2.7348,0.3260
theta,Theta Forecaster,1.7319,1.4771,13.1343,15.7314,0.2684,0.2698,-4.2090,0.0080
arima,ARIMA,1.7847,1.5626,13.2031,16.0011,0.3032,0.3049,-5.6593,0.0140


Transformation Pipeline and Model Successfully Saved
>>> [NFLX] Model tersimpan di: ../Models/NFLX_model_terbaik
>>> [SUKSES] NFLX selesai diproses.

RINGKASAN PEMODELAN SELURUH SAHAM


,Ticker,Model Terbaik,MAE,RMSE,R2
0,AAPL,ETS,21.2915,24.0332,-3.4215
1,MSFT,Auto ARIMA,24.5648,29.6979,-0.5229
2,GOOGL,Huber w/ Cond. Deseasonalize & Detrending,14.1511,17.7863,-1.4358
3,AMZN,Huber w/ Cond. Deseasonalize & Detrending,19.4222,21.5106,-2.8492
4,NVDA,Theta Forecaster,10.7715,13.6142,-2.2951
5,META,Exponential Smoothing,71.5023,88.1713,-1.6202
6,TSLA,Orthogonal Matching Pursuit w/ Cond. Deseasona...,72.6846,85.3131,-1.7613
7,AVGO,Auto ARIMA,16.0093,19.4717,-1.1968
8,AMD,Decision Tree w/ Cond. Deseasonalize & Detrending,25.6188,29.0936,-2.2785
9,ORCL,Auto ARIMA,13.8870,15.8409,-1.6278
